In [21]:
import os

files_to_delete = [
    "cease_requests.db",
    "archive_log.txt",
    "audit_log.json"
]

# delete system files
for file in files_to_delete:
    if os.path.exists(file):
        os.remove(file)

# delete uploaded PDFs
for file in os.listdir():

    if file.endswith(".pdf"):
        os.remove(file)

print("System reset complete")

System reset complete


In [2]:
!pip install langchain
!pip install langchain-google-genai
!pip install google-generativeai
!pip install pypdf
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.2 MB/s eta 0:00:00


In [3]:
import os
import json
import sqlite3
import time
from datetime import datetime

from pypdf import PdfReader
from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI

import ipywidgets as widgets
from IPython.display import display

In [40]:
os.environ["GOOGLE_API_KEY"] = ""

In [41]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [42]:
class DocumentClassification(BaseModel):

    decision: str = Field(description="Cease, Uncertain, or Irrelevant")

    confidence: float = Field(description="Confidence score between 0 and 1")

    explanation: str = Field(description="Reason for the classification")

In [43]:
structured_llm = llm.with_structured_output(DocumentClassification)

In [44]:
def load_pdf(file_path):

    reader = PdfReader(file_path)

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text

    return text.strip()

In [45]:
#from langchain_core.messages import HumanMessage

def classify_document(text):

    prompt = f"""
You are a legal compliance AI.

Classify the document into one category:

Cease → Valid cease & desist request
Uncertain → Needs manual review
Irrelevant → Not a cease request

IMPORTANT RULES:
- Return ONLY JSON
- Do NOT write explanations outside JSON
- Do NOT ask questions

Also provide:
- confidence score between 0 and 1
- short explanation for your classification

Return response strictly in JSON format:

{{"decision": "Cease | Uncertain | Irrelevant", "confidence": 0.0, "explanation": "reason for classification"}}

Document:
{text[:3000]}
"""

    result = structured_llm.invoke(prompt)

    return {
        "decision": result.decision,
        "confidence": result.confidence,
        "explanation": result.explanation
    }

In [46]:
conn = sqlite3.connect("cease_requests.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS cease_requests(
id INTEGER PRIMARY KEY,
Date_of_document_received TEXT,
Document_Name TEXT,
Extracted_details TEXT
)
""")

conn.commit()
conn.close()

In [48]:
def database_agent(doc_name, details):

    conn = sqlite3.connect("cease_requests.db")
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO cease_requests(Date_of_document_received, Document_Name,
    Extracted_details)
    VALUES (?, ?, ?)
    """,(str(datetime.now()), doc_name, details))

    conn.commit()
    conn.close()

In [50]:
from datetime import datetime
import json

def archive_agent(doc_name):

    log = {
        "Date of document received": str(datetime.now()),
        "Document name": doc_name
    }

    with open("archive_log.txt", "a+") as f:
        f.write(json.dumps(log) + "\n")

In [51]:
def audit_agent(doc_name, decision, confidence, explanation):

    log = {
        "Date of document received": str(datetime.now()),
        "Document name": doc_name,
        "Classification": decision,
        "Confidence": confidence,
        "Explanation": explanation
    }

    with open("audit_log.json","a") as f:
        f.write(json.dumps(log) + "\n")

In [52]:
def human_review_agent(text, file_path):

    """preview = widgets.Textarea(
        value=text[:500],
        description='Preview:',
        disabled=True,
        layout=widgets.Layout(width='600px', height='120px')
    )"""

    dropdown = widgets.Dropdown(
        options=["Cease", "Irrelevant"],
        description="Decision:"
    )

    submit_button = widgets.Button(description="Submit Decision")

    output = widgets.Output()

    def on_submit(b):

        decision = dropdown.value

        with output:
            output.clear_output()
            print("Human Final Decision:", decision)

        # audit log
        audit_agent(file_path, decision, "Human", "Human review override")

        if decision == "Cease":

            database_agent(file_path, text)

            print("Stored in database after human review")

        else:

            archive_agent(file_path)

            print("Archived after human review")

    submit_button.on_click(on_submit)

    display(dropdown, submit_button, output)

In [53]:
def process_document(file_path):

    print("\n-------------------------------")
    print("Processing:", file_path)
    print("-------------------------------")

     # Load document
    text = load_pdf(file_path)

    # Classify document
    result = classify_document(text)

    decision = result["decision"]
    confidence = result["confidence"]
    explanation = result["explanation"]

    print("AI Classification:", decision)
    print("Confidence:", confidence)
    print("Explanation:", explanation)

    audit_agent(file_path, decision, confidence, explanation)

    if decision == "Cease":

        database_agent(file_path, text)

    elif decision == "Irrelevant":

        archive_agent(file_path)

    else:

        print("\nSending to Human Review")

        human_review_agent(text, file_path)

In [54]:
from google.colab import files

uploaded = files.upload()

Saving notice_1.pdf to notice_1.pdf
Saving notice_2.pdf to notice_2.pdf
Saving notice_3.pdf to notice_3.pdf
Saving notice_4.pdf to notice_4.pdf
Saving notice_5.pdf to notice_5.pdf


In [55]:
for file_name in uploaded.keys():

    process_document(file_name)


-------------------------------
Processing: notice_1.pdf
-------------------------------
AI Classification: Uncertain
Confidence: 0.9
Explanation: The document is empty, making it impossible to classify as a cease and desist request or irrelevant. It requires manual review to determine its content and purpose.

Sending to Human Review


Dropdown(description='Decision:', options=('Cease', 'Irrelevant'), value='Cease')

Button(description='Submit Decision', style=ButtonStyle())

Output()


-------------------------------
Processing: notice_2.pdf
-------------------------------
AI Classification: Uncertain
Confidence: 0.95
Explanation: The document is empty, making it impossible to classify as a cease and desist request or irrelevant. It requires manual review to provide content.

Sending to Human Review


...


-------------------------------
Processing: notice_5.pdf
-------------------------------
AI Classification: Uncertain
Confidence: 0.95
Explanation: The document is empty, making it impossible to classify as a cease and desist request or irrelevant. It requires manual review to provide content.

Sending to Human Review


Dropdown(description='Decision:', options=('Cease', 'Irrelevant'), value='Cease')

Button(description='Submit Decision', style=ButtonStyle())

Output()

Archived after human review
Archived after human review
Archived after human review
Stored in database after human review
Stored in database after human review
Archived after human review
Stored in database after human review


In [59]:
with open("audit_log.json","r") as f:
    print(f.read())

{"Date of document received": "2026-03-26 11:04:36.725223", "Document name": "bw_doc_1.pdf", "Classification": "Uncertain", "Confidence": 0.9, "Explanation": "The document is empty, making it impossible to classify as a cease and desist request or irrelevant. It requires manual review to determine its content and purpose."}
{"Date of document received": "2026-03-26 11:04:38.158033", "Document name": "bw_doc_2.pdf", "Classification": "Uncertain", "Confidence": 0.9, "Explanation": "The document is empty, making it impossible to classify as a cease and desist request or irrelevant. It requires manual review to determine its content and purpose."}
{"Date of document received": "2026-03-26 11:04:39.561239", "Document name": "bw_doc_3.pdf", "Classification": "Uncertain", "Confidence": 0.95, "Explanation": "The document is empty, making it impossible to classify as a cease and desist request or irrelevant. It requires manual review to provide content."}
{"Date of document received": "2026-03-

In [60]:
if os.path.exists("archive_log.txt"):
    print(open("archive_log.txt").read())

{"Date of document received": "2026-03-26 11:05:02.152602", "Document name": "bw_doc_3.pdf"}
{"Date of document received": "2026-03-26 11:05:21.134251", "Document name": "bw_doc_4.pdf"}
{"Date of document received": "2026-03-26 11:05:27.042827", "Document name": "bw_doc_5.pdf"}
{"Date of document received": "2026-03-26 11:10:14.728431", "Document name": "notice_1.pdf"}
{"Date of document received": "2026-03-26 11:10:18.846568", "Document name": "notice_2.pdf"}
{"Date of document received": "2026-03-26 11:10:33.126463", "Document name": "notice_3.pdf"}
{"Date of document received": "2026-03-26 11:24:51.455704", "Document name": "notice_1.pdf"}



In [58]:
conn = sqlite3.connect("cease_requests.db")

cursor = conn.cursor()

cursor.execute("SELECT * FROM cease_requests")

cursor.fetchall()

[(1, '2026-03-26 11:04:53.502336', 'bw_doc_1.pdf', ''),
 (2, '2026-03-26 11:04:57.198833', 'bw_doc_2.pdf', ''),
 (3, '2026-03-26 11:10:38.248385', 'notice_4.pdf', ''),
 (4, '2026-03-26 11:10:42.080865', 'notice_5.pdf', '')]